# 📖 AudiobookMaker Colab WebUI

Welcome to the Google Colab Notebook for **AudiobookMaker**! This notebook sets up and runs the AudiobookMaker application.

Since AudiobookMaker uses both Python and a high-performance Rust PyO3 acceleration module (`audiobook_rust`), this notebook will:
1. Install the Rust compiler toolchain.
2. Compile the Rust PyO3 modules on the Colab Linux VM.
3. Install all Python dependencies (including PyTorch with CUDA & `qwen-tts`).
4. Launch the FastAPI orchestration server in the background.
5. Launch the Gradio Web UI with a public shareable link.

### ⚡ Select GPU Runtime (CRITICAL)
Before running any cells, make sure you are using a GPU runtime so that the TTS model runs fast.
- Go to **Runtime** (top menu) -> **Change runtime type**.
- Select **T4 GPU** (or any other available GPU) from the dropdown.
- Click **Save**.

In [ ]:
# Check if GPU accelerator is assigned
!nvidia-smi

### 📁 Clone Repository and Setup Workspace

In [ ]:
import os
if not os.path.exists("audiobook_rust"):
    print("Cloning AudiobookMaker repository...")
    !git clone https://github.com/MSpider3/AudiobookMaker.git
    %cd AudiobookMaker
else:
    print("Already in AudiobookMaker project directory!")

### 🦀 Install Rust Compiler Toolchain

In [ ]:
print("Installing Rustup and Cargo...")
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

# Add Cargo binaries to Python path
import os
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.cargo/bin")

# Verify Rust tools are available
!cargo --version
!rustc --version

### 📦 Install System dependencies and Python requirements

In [ ]:
# Install FFmpeg
!apt-get update && apt-get install -y ffmpeg

# Install dependencies from requirements.txt
print("Installing requirements (this takes a few minutes)...")
!pip install -r requirements.txt

# Install maturin (needed to build python-rust PyO3 modules)
!pip install maturin

### ⚙️ Compile Rust PyO3 Module

In [ ]:
print("Compiling audiobook_rust module using maturin...")
%cd audiobook_rust
!maturin develop -r
%cd ..

# Verify import works
try:
    import audiobook_rust
    print("Success! audiobook_rust compiled and imported successfully. 5.5x faster mastering is enabled.")
except ImportError as e:
    print("Error compiling Rust code:", e)
    print("Fallback: The app will run, but with slower pure-Python fallback code.")

### 🚀 Start FastAPI Orchestration Server in Background

In [ ]:
import subprocess
import time
import requests

print("Starting start_api.py in the background...")
api_proc = subprocess.Popen(["python", "start_api.py"])

# Check health for up to 30 seconds
for i in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=1.0)
        if r.status_code == 200 and r.json().get("status") == "ok":
            print("FastAPI Server is healthy and running!")
            break
    except Exception:
        pass
else:
    print("Warning: FastAPI healthcheck timed out, Python Gradio UI will use local processing fallback.")

### 🎨 Launch Gradio Web Interface
Run this cell to start the UI. It will generate a public **`gradio.live`** link for you. Click that link to open the AudiobookMaker interface!

In [ ]:
from app import build_app

demo = build_app()
# share=True produces a public shareable URL, server_name='0.0.0.0' allows external routing
demo.launch(
    server_name="0.0.0.0",
    share=True,
    show_error=True
)